In [89]:
import pandas as pd
from pathlib import Path    
from statsmodels.stats.multitest import multipletests
from scipy import stats
import numpy as np

In [ ]:
result_path = Path("../results/downstream_task")
bias_types = ["less_positive_class"]
metrics = ["AUROC"]
less_bias_strengths = ["0.1"]
method_name_replacer = {"mrs-forest": "MRS",  
                        "fw-mrs-temperature": "FW-MRS",
                        "fw-mrs-temperature-svm": "FW-MRS$_{SVM}$",
                        }
data_set_replacer = {
                    "folktables_employment": "Employment",
                    "folktables_income": "Income",
                    "breast_cancer": "Breast Cancer",
                    "hr_analytics": "HR Analytic",
                    "loan_prediction": "Loan",
                    "diabetes": "Diabetes",
                    "german_credit": "German Credit",
                    "bank_marketing": "Bank Marketing"
                    }

In [91]:
method_pairs = []
for other_method in ("fw-mrs-temperature", "fw-mrs-temperature-svm"):
    method_pairs.append(("mrs-forest", other_method))
method_pairs

[('mrs-forest', 'fw-mrs-temperature'),
 ('mrs-forest', 'fw-mrs-temperature-svm')]

In [92]:
aurocs = []
auprcs = []
dict_list = []
for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
        for method in method_name_replacer.keys():
            for bias_strength in less_bias_strengths:
                json_directory = result_path / dataset / bias_type /  bias_strength/ method / "classification_results"
                auroc_file = pd.read_json(str(json_directory / "rf_auroc_list.json"))
                dict_list.append(
                    {
                        "Method": method,
                        "Data Set": dataset,
                        "AUROC": auroc_file.values,
                        "Bias Type": bias_type,
                        "Bias Strength": bias_strength
                    }
                                  )
result_df = pd.DataFrame(data=dict_list)

In [93]:
result_df.explode("AUROC")

,Method,Data Set,AUROC,Bias Type,Bias Strength
0,mrs-forest,breast_cancer,[0.9899344569288391],less_positive_class,0.1
0,mrs-forest,breast_cancer,[0.9932116104868911],less_positive_class,0.1
0,mrs-forest,breast_cancer,[0.9926264044943821],less_positive_class,0.1
0,mrs-forest,breast_cancer,[0.9792015300023901],less_positive_class,0.1
0,mrs-forest,breast_cancer,[0.9862689393939391],less_positive_class,0.1
...,...,...,...,...,...
11,fw-mrs-temperature-svm,diabetes,[0.7447608173362831],less_positive_class,0.1
11,fw-mrs-temperature-svm,diabetes,[0.796076262518137],less_positive_class,0.1
11,fw-mrs-temperature-svm,diabetes,[0.753808963105687],less_positive_class,0.1
11,fw-mrs-temperature-svm,diabetes,[0.77796807726159],less_positive_class,0.1


In [94]:
def corrected_t_test(first_values, second_values, n_folds=5.0):
    differences = first_values - second_values
    mean_differences = np.mean(differences)
    var_differences = np.var(differences)
    train_size = n_folds - 1.0
    test_size = 1.0 
    correction_factor = (1.0 / len(differences)) + (test_size / train_size)
    return mean_differences / (np.sqrt(correction_factor * var_differences))

In [95]:
p_values = []
for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
            for bias_strength in less_bias_strengths:
                for first_method_name, second_metric_name in method_pairs:
                    for metric in metrics:
                        first_metrics = result_df.loc[(result_df["Method"]==first_method_name) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & (result_df["Data Set"]==dataset)][metric].values[0]
                        second_metrics = result_df.loc[(result_df["Method"]==second_metric_name) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & (result_df["Data Set"]==dataset)][metric].values[0]
                        t_statistic = corrected_t_test(np.squeeze(first_metrics), np.squeeze(second_metrics))
                        p_values.append(stats.t.sf(np.abs(t_statistic), len(first_metrics-1)) * 2)
corrected_p_values = multipletests(p_values, method="fdr_bh")

In [96]:
i = 0
for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
            for bias_strength in less_bias_strengths:
                for first_method_name, second_metric_name in method_pairs:
                    for metric in metrics:
                        print(f"p value for {metric}, {dataset}, {bias_type}, {bias_strength}, {first_method_name},\
{second_metric_name} is: {corrected_p_values[0][i]}")
                        i += 1

p value for AUROC, breast_cancer, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature is: False
p value for AUROC, breast_cancer, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature-svm is: False
p value for AUROC, hr_analytics, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature is: False
p value for AUROC, hr_analytics, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature-svm is: False
p value for AUROC, loan_prediction, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature is: False
p value for AUROC, loan_prediction, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature-svm is: False
p value for AUROC, diabetes, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature is: False
p value for AUROC, diabetes, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature-svm is: False
